In [16]:
import pandas as pd

file_path = 'datasets/titanic.csv'
df = pd.read_csv(file_path)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# Missing value count + percentage
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing[missing['missing_count'] > 0])

# Duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

# Data types
print(df.dtypes)

# Unique values for categorical columns
for col in ['Sex', 'Embarked', 'Pclass']:
    print(f"{col}: {df[col].unique()}")


          missing_count  missing_pct
Age                 177        19.87
Cabin               687        77.10
Embarked              2         0.22
Duplicate rows: 0
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object
Sex: <StringArray>
['male', 'female']
Length: 2, dtype: str
Embarked: <StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str
Pclass: [3 1 2]


**Which columns have nulls and how bad is it?**

Age is having null in 20% of the records. and Cabin is having null in 687 records(~77%) - critical

**Are there any duplicates?**

no

**Any columns with wrong data types?**

Age is in floater which is wrong - must be int datatype. 

### Self notes for filling missing values:

 - if the missing value count is less, then fill with median/mean. Median is safer for large dataset in terms of computation time.
 - If the missing count is significantly low, go with mode - frequently occurring value.
 - If a column is composed of almost everything null, it doesnt make it useful enough. It is better off dropped.

In [3]:
# Age has ~20% missing — fill with median (median is safer than mean with outliers)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Verify
print(f"Age nulls remaining: {df['Age'].isnull().sum()}")

Age nulls remaining: 0


In [4]:
# Embarked has 2 missing — fill with most frequent value
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print(f"Embarked nulls remaining: {df['Embarked'].isnull().sum()}")

Embarked nulls remaining: 0


In [5]:
# Cabin has 77% missing — not worth keeping
print(f"Cabin missing: {df['Cabin'].isnull().mean() * 100:.1f}%")
df = df.drop(columns=['Cabin'])

Cabin missing: 77.1%


In [6]:
# 1. What % of Cabin was missing? Was dropping the right call? - the column doesnt produce enough inputs to be useful for modeling, and it has a very high missing rate, so dropping is reasonable.
# 2. Why median over mean for Age? (write your answer in markdown) - The median is less affected by outliers than the mean. If there are a few very old passengers, they could skew the mean age upwards, while the median would give a better representation of the typical passenger's age.
# 3. Check nulls again after all fills — should be 0 or close
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64

In [10]:
# Pclass is numeric but it's actually a category
df['Pclass'] = df['Pclass'].astype('category')

# Survived is numeric but it's binary — treat as category
df['Survived'] = df['Survived'].astype('category')

# Verify
print(df.dtypes)

PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex                 str
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Embarked            str
dtype: object


In [11]:
# Standardize Sex column — strip whitespace, lowercase
df['Sex'] = df['Sex'].str.strip().str.lower()

# Extract title from Name — this is feature engineering basics
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
print(df['Title'].value_counts())

# Simplify rare titles
df['Title'] = df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don',
     'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

print(df['Title'].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
Title
Mr              517
Miss            185
Mrs             126
Master           40
Rare             22
the Countess      1
Name: count, dtype: int64


In [12]:
# Feature 1: Family size (SibSp = siblings/spouses, Parch = parents/children)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves

# Feature 2: Is alone?
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Quick check — did alone passengers survive less?
print(df.groupby('IsAlone')['Survived'].value_counts())

IsAlone  Survived
0        1           179
         0           175
1        0           374
         1           163
Name: count, dtype: int64


### Create a 3rd feature of your own — ideas:
 - FarePerPerson = Fare / FamilySize
 - AgeGroup = bins of Age (child / adult / senior)
 - Write a markdown cell explaining what your feature captures

In [14]:
df['FarePerPerson'] = df['Fare'] / df['FamilySize']
print(df['FarePerPerson'].head())
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 80], labels=['Child', 'Teen', 'Adult', 'Senior', 'Elder'])
print(df['AgeGroup'].value_counts())

0     3.62500
1    35.64165
2     7.92500
3    26.55000
4     8.05000
Name: FarePerPerson, dtype: float64
AgeGroup
Adult     535
Senior    195
Teen       70
Child      69
Elder      22
Name: count, dtype: int64


In [17]:
def clean_titanic(df):
    df = df.copy()  # never mutate the original
    
    # Drop high-null columns
    df = df.drop(columns=['Cabin'])
    
    # Fill missing values
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # Fix dtypes
    df['Pclass'] = df['Pclass'].astype('category')
    df['Survived'] = df['Survived'].astype('category')
    
    # String cleaning
    df['Sex'] = df['Sex'].str.strip().str.lower()
    
    # Feature engineering
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    return df

df_clean = clean_titanic(df)
print(df_clean.shape)
print(df_clean.isnull().sum())

(891, 13)
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
FamilySize     0
IsAlone        0
dtype: int64
